In [1]:
import os, shutil, tempfile
os.environ["HYDRA_FULL_ERROR"]="1"

import warnings
from collections.abc import Sequence
import subprocess

# def performance_tweaks():
#     import torch
#     torch.backends.cudnn.benchmark = True
#     torch.backends.cudnn.deterministic = False
#     torch.use_deterministic_algorithms(False)
#     torch.set_float32_matmul_precision('high')
#     torch.jit.enable_onednn_fusion(True)
#     torch.autograd.set_detect_anomaly(False, False) # type:ignore
#     torch.autograd.profiler.profile(False) # type:ignore
#     torch.autograd.profiler.emit_nvtx(False) # type:ignore

# performance_tweaks() # this has 0 effect btw

PYTHON_BIN = "/home/jj/miniconda3/envs/mapperatorinator/bin/python"
PYTHON_BIN = "/var/mnt/issd/files 2/programming/libs/mapperatorinator/Mapperatorinator/.venv/bin/python"


# ---------------------------------------------------------------------------- #
#                                  definitions                                 #
# ---------------------------------------------------------------------------- #

# [
#     "/home/jj/miniconda3/envs/mapperatorinator/bin/python",
#     "inference.py",
#     "-cn",
#     "v30",
#     "audio_path='/var/mnt/ssd/Files/Documents/Музыка/a.mp3'",
#     "output_path='/var/mnt/ssd/Files/Programming/libs/mapperatorinator'",
#     "gamemode=0",
#     "difficulty=5",
#     "year=2023",
#     "hp_drain_rate=5",
#     "circle_size=4",
#     "overall_difficulty=8",
#     "approach_rate=9",
#     "slider_multiplier=1.4",
#     "slider_tick_rate=1",
#     "keycount=4",
#     "cfg_scale=1.0",
#     "temperature=0.9",
#     "top_p=0.9",
#     "export_osz=true",
#     "add_to_beatmap=false",
#     "hitsounded=true",
#     "super_timing=true",
# ]

def _to_string_seq(x):
    if x is None: return ()
    if isinstance(x, str): return (x, )
    return x

def get_command(
    name: str,
    input:str,
    output:str,
    difficulty: float = 5,
    mapper_id: int | str | None = None,
    descriptors: str | Sequence[str] | None = (), # ["jump aim", "clean"]
    negative_descriptors: str | Sequence[str] | None = (), # from discord ["improvisation","tech","bursts"]
    cfg_scale: float | None = None, # 1 # Scale of classifier-free guidance
    temperature: float | None = None, # 0.9
    top_p: float | None = None, # 0.9
    year: int | None = None, # 2023

    super_timing: bool | None = None,

    approach_rate: float | None = None, # 9
    circle_size: float | None = None, # 4
    hp_drain_rate: float | None = None, # 5
    overall_difficulty: float | None = None, # 8
    slider_multiplier: float | None = None, # 1.4,
    slider_tick_rate: float | None = None, # 1

    timer_bpm_threshold: float | None = None, # 0.7
    generate_positions: bool | None = None, # false #  Use diffusion to generate object positions
    refine_iters: int | None = None, # 10, Number of diffusion refinement iterations

    lora_path: str | None = None,
):
    descriptors = _to_string_seq(descriptors)
    negative_descriptors = _to_string_seq(negative_descriptors)

    commands = [
        PYTHON_BIN,
        "inference.py",
        "-cn", "v30", # load v30 config
        "device='auto'",
        # "compile=false",
        f"audio_path='{input}'",
        f"output_path='{output}'",
        "gamemode=0",
        f"difficulty={difficulty}",
        "export_osz=true",
        "add_to_beatmap=false",
        "hitsounded=true",
    ]

    sq = "'"
    if len(descriptors) > 0:
        s = f"'{f'{sq},{sq}'.join(descriptors)}'"
        commands.append(f'descriptors=[{s}]')

    if len(negative_descriptors) > 0:
        s = f"'{f'{sq},{sq}'.join(negative_descriptors)}'"
        commands.append(f'negative_descriptors=[{s}]')

    if lora_path is not None: commands.insert(6, f"lora_path='{lora_path}'")
    if year is not None: commands.append(f"year={year}")
    if mapper_id is not None: commands.append(f"mapper_id={mapper_id}")
    if super_timing is not None: commands.append(f"super_timing={str(bool(super_timing)).lower()}")

    if cfg_scale is not None: commands.append(f"cfg_scale={cfg_scale}")
    if temperature is not None: commands.append(f"temperature={temperature}")
    if top_p is not None: commands.append(f"top_p={top_p}")

    if approach_rate is not None: commands.append(f"approach_rate={approach_rate}")
    if circle_size is not None: commands.append(f"circle_size={circle_size}")
    if hp_drain_rate is not None: commands.append(f"hp_drain_rate={hp_drain_rate}")
    if overall_difficulty is not None: commands.append(f"overall_difficulty={overall_difficulty}")
    if slider_multiplier is not None: commands.append(f"slider_multiplier={slider_multiplier}")
    if slider_tick_rate is not None: commands.append(f"slider_tick_rate={slider_tick_rate}")
    if timer_bpm_threshold is not None: commands.append(f"timer_bpm_threshold={timer_bpm_threshold}")
    if generate_positions is not None: commands.append(f"generate_positions={str(bool(generate_positions)).lower()}")
    if refine_iters is not None: commands.append(f"refine_iters={refine_iters}")

    # determine artist title
    filename = '.'.join(os.path.basename(input).split('.')[:-1])
    if ' - ' in filename:
        artist, title = filename.split(' - ')
    else:
        artist = ''
        title = filename

    commands.append(f"artist='{artist}'")
    commands.append(f"title='{title}'")
    commands.append(f"creator='Mapperatorinator ({name})'")

    return commands


# ---------------------------------------------------------------------------- #
#                                    presets                                   #
# ---------------------------------------------------------------------------- #
presets = []

# ---------------------------------- default --------------------------------- #
presets.append(lambda input,output: get_command(
    input=input,
    output=output,
    name='default',
    year=2023,
    super_timing=True,
    approach_rate=10,
    difficulty=5,
))

# ---------------------------------- Kroytz ---------------------------------- #
presets.append(lambda input,output: get_command(
    input=input,
    output=output,
    name="Kroytz",
    descriptors=["jump aim", "clean"],
    mapper_id=2339768,
    cfg_scale=1.3,
    year=2023,
    super_timing=True,
    approach_rate=10,
    difficulty=5.5,
)) # has jumps on sidewinder second drop, generally good jumps

# ---------------------------------- Sotarks --------------------------------- #
presets.append(lambda input,output: get_command(
    input=input,
    output=output,
    name="Sotarks",
    descriptors=["jump aim", "clean"],
    mapper_id=4452992,
    cfg_scale=1.3,
    year=2023,
    super_timing=True,
    approach_rate=10,
    difficulty=5.5,
)) # good patterns, jumps, no insane parts


# -------------------------------- LoRA-Kroytz-default ------------------------------- #
presets.append(lambda input,output: get_command(
    input=input,
    output=output,
    name="LoRA-Kroytz",
    year=2023,
    super_timing=True,
    approach_rate=10,
    difficulty=5.5,
    lora_path="OliBomby/Mapperatorinator-v30-LoRA-Kroytz",
))

# -------------------------------- LoRA-Kroytz-Kroytz ------------------------------- #
presets.append(lambda input,output: get_command(
    input=input,
    output=output,
    name="LoRA-Kroytz Kroytz",
    descriptors=["jump aim", "clean"],
    mapper_id=2339768,
    cfg_scale=1.3,
    year=2023,
    super_timing=True,
    approach_rate=10,
    difficulty=5.5,
    lora_path="OliBomby/Mapperatorinator-v30-LoRA-Kroytz",
))


# -------------------------------- LoRA-aim-control ------------------------------- #
presets.append(lambda input, output: get_command(
    input=input,
    output=output,
    name="LoRA-aim-control",
    year=2023,
    super_timing=True,
    approach_rate=10,
    difficulty=5.5,
    lora_path="mjoink/Mapperatorinator-v30-LoRA-aim-control",
))

# -------------------------------- LoRA-Arles ------------------------------- #
presets.append(lambda input, output: get_command(
    input=input,
    output=output,
    name="LoRA-Arles",
    year=2023,
    super_timing=True,
    approach_rate=10,
    difficulty=5.5,
    lora_path="mouceen/Mapperatorinator-v30-LoRA-Arles-v1",
))


# ---------------------------------------------------------------------------- #
#                                postprocessing                                #
# ---------------------------------------------------------------------------- #
def packgod(folder, out_dir):
    files = os.listdir(folder)
    osz = ''
    osus = []

    # find osz
    for file in files:
        path = os.path.join(folder, file)
        if path.endswith('.osz'):
            if osz != '': os.remove(path)
            else: osz = path
        elif file.endswith('.osu'):
            osus.append(os.path.join(folder, file))

    # for osu_file in osus:
        # with open(osu_file, 'r+') as f:
        #     osu = f.read()
        #     osu = osu.replace("Version:Mapperatorinator V30", "Version:Mapperatorinator V30\nBeatmapID:0\nBeatmapSetID:-1")


        #     # write
        #     f.seek(0)
        #     f.write(osu)
        #     f.truncate()

    filename = 'unknown'

    # unpack osz
    with tempfile.TemporaryDirectory() as temp:
        shutil.unpack_archive(osz, temp, 'zip')
        for f in os.listdir(temp):
            path = os.path.join(temp, f)
            if path.endswith('.osu'): os.remove(path)
            else: filename = '.'.join(f.split('.')[:-1])


        for osu_file in osus:
            shutil.move(osu_file, temp)

        shutil.make_archive(f"{filename}", 'zip', root_dir=temp)
        shutil.move(f"{filename}.zip", os.path.join(out_dir, f"{filename}.osz"))



# ---------------------------------------------------------------------------- #
#                                      run                                     #
# ---------------------------------------------------------------------------- #
def run(file: str):
    if not os.path.exists(file):
        raise FileNotFoundError(file)
    
    os.chdir("/var/mnt/issd/files 2/programming/libs/mapperatorinator/Mapperatorinator")
    with tempfile.TemporaryDirectory() as tempdir:

        for preset in presets:
            subprocess.run(preset(file, tempdir))

        packgod(tempdir, "/var/mnt/ssd/Files/Programming/libs/mapperatorinator/out")


### random song suggester AGI

In [8]:
SONGS_DIR = "/var/mnt/issd/files/music/tracks"
import random
os.path.join(SONGS_DIR, random.choice(os.listdir("/var/mnt/issd/files/music/tracks")))

'/var/mnt/issd/files/music/tracks/Hudson Lee - Axon.mp3'

# Don't forget that script directory is changed!
Use absolute paths only


what I meant is that I use os.chdir so script path is changed, and therefore relative paths become wrong

In [ ]:
run("/var/mnt/issd/files/music/tracks/Hudson Lee - Axon.mp3")

mismatched input 't' expecting <EOF>
See https://hydra.cc/docs/1.2/advanced/override_grammar/basic for details

Set the environment variable HYDRA_FULL_ERROR=1 for a complete stack trace.
mismatched input 't' expecting <EOF>
See https://hydra.cc/docs/1.2/advanced/override_grammar/basic for details

Set the environment variable HYDRA_FULL_ERROR=1 for a complete stack trace.
mismatched input 't' expecting <EOF>
See https://hydra.cc/docs/1.2/advanced/override_grammar/basic for details

Set the environment variable HYDRA_FULL_ERROR=1 for a complete stack trace.
mismatched input 't' expecting <EOF>
See https://hydra.cc/docs/1.2/advanced/override_grammar/basic for details

Set the environment variable HYDRA_FULL_ERROR=1 for a complete stack trace.
mismatched input 't' expecting <EOF>
See https://hydra.cc/docs/1.2/advanced/override_grammar/basic for details

Set the environment variable HYDRA_FULL_ERROR=1 for a complete stack trace.
mismatched input 't' expecting <EOF>
See https://hydra.cc/do

In [3]:
run("/var/mnt/ssd/Files/Documents/Музыка/Tracks/myai - Rhythm Era.mp3")

Using CUDA for inference (auto-selected).
Random seed: 65461
Model loaded: OliBomby/Mapperatorinator-v30 on device cuda
Using HP drain rate 5.0
Using circle size 4.0
Using overall difficulty 8.0
Using slider multiplier 1.4
Using slider tick rate 1.0
Generating map


100%|██████████| 52/52 [03:00<00:00,  3.47s/it]


Generated beatmap saved to /tmp/tmp21ff_vhk/beatmapc96ce655ee5446699a7c58e0eeb4f81a.osu
Generated .osz saved to /tmp/tmp21ff_vhk/beatmap05ecaa01e9b44bd4b9603c8451394afc.osz
Using CUDA for inference (auto-selected).
Random seed: 15768
Model loaded: OliBomby/Mapperatorinator-v30 on device cuda
Using HP drain rate 5.0
Using circle size 4.0
Using overall difficulty 8.0
Using slider multiplier 1.4
Using slider tick rate 1.0
Generating map


100%|██████████| 52/52 [05:27<00:00,  6.30s/it]


Generated beatmap saved to /tmp/tmp21ff_vhk/beatmapd2f74f630cc545ff851233c64b3536b4.osu
Generated .osz saved to /tmp/tmp21ff_vhk/beatmap68033c7857cb45edb83f63f2078b736b.osz
Using CUDA for inference (auto-selected).
Random seed: 23212
Model loaded: OliBomby/Mapperatorinator-v30 on device cuda
Using HP drain rate 5.0
Using circle size 4.0
Using overall difficulty 8.0
Using slider multiplier 1.4
Using slider tick rate 1.0
Generating map


100%|██████████| 52/52 [04:23<00:00,  5.06s/it]


Generated beatmap saved to /tmp/tmp21ff_vhk/beatmap9eed84a0333744a880ca19a49e2118bc.osu
Generated .osz saved to /tmp/tmp21ff_vhk/beatmapdab37f07134645e2baca5b14e0b38339.osz


In [4]:
run("/var/mnt/ssd/Files/Documents/Музыка/Tracks/myai - white.mp3")

Using CUDA for inference (auto-selected).
Random seed: 23684
Model loaded: OliBomby/Mapperatorinator-v30 on device cuda
Using HP drain rate 5.0
Using circle size 4.0
Using overall difficulty 8.0
Using slider multiplier 1.4
Using slider tick rate 1.0
Generating map


100%|██████████| 55/55 [03:06<00:00,  3.40s/it]


Generated beatmap saved to /tmp/tmpjfshp9du/beatmapddc6c256345c4488a263adaa9b99bcb4.osu
Generated .osz saved to /tmp/tmpjfshp9du/beatmape2f3e948677742c7a38a869034852906.osz
Using CUDA for inference (auto-selected).
Random seed: 34182
Model loaded: OliBomby/Mapperatorinator-v30 on device cuda
Using HP drain rate 5.0
Using circle size 4.0
Using overall difficulty 8.0
Using slider multiplier 1.4
Using slider tick rate 1.0
Generating map


100%|██████████| 55/55 [04:54<00:00,  5.36s/it]


Generated beatmap saved to /tmp/tmpjfshp9du/beatmap93293094260047bd96365fad9d12e702.osu
Generated .osz saved to /tmp/tmpjfshp9du/beatmap2fddb3fc3efb4f5aaa98f94b88b9a3c8.osz
Using CUDA for inference (auto-selected).
Random seed: 2763
Model loaded: OliBomby/Mapperatorinator-v30 on device cuda
Using HP drain rate 5.0
Using circle size 4.0
Using overall difficulty 8.0
Using slider multiplier 1.4
Using slider tick rate 1.0
Generating map


100%|██████████| 55/55 [04:27<00:00,  4.87s/it]


Generated beatmap saved to /tmp/tmpjfshp9du/beatmap08decc6dec4744d198afb9a515ee8426.osu
Generated .osz saved to /tmp/tmpjfshp9du/beatmap77ef0667a93c4e068b4ec5c285e46a50.osz


In [5]:
run("/var/mnt/ssd/Files/Documents/Музыка/Tracks/myai - another stuff.mp3")

Using CUDA for inference (auto-selected).
Random seed: 22103
Model loaded: OliBomby/Mapperatorinator-v30 on device cuda
Using HP drain rate 5.0
Using circle size 4.0
Using overall difficulty 8.0
Using slider multiplier 1.4
Using slider tick rate 1.0
Generating map


100%|██████████| 31/31 [03:21<00:00,  6.51s/it]


Generated beatmap saved to /tmp/tmph7p21vbg/beatmap009bb57935b44798842699532a0d3899.osu
Generated .osz saved to /tmp/tmph7p21vbg/beatmap25a52a73ed92431ab4ded8a880e61d4f.osz
Using CUDA for inference (auto-selected).
Random seed: 48364
Model loaded: OliBomby/Mapperatorinator-v30 on device cuda
Using HP drain rate 5.0
Using circle size 4.0
Using overall difficulty 8.0
Using slider multiplier 1.4
Using slider tick rate 1.0
Generating map


100%|██████████| 31/31 [04:58<00:00,  9.62s/it]


Generated beatmap saved to /tmp/tmph7p21vbg/beatmap42324ef19b3d4053b674e11bf9a6fc30.osu
Generated .osz saved to /tmp/tmph7p21vbg/beatmap07dd67b83cdf4af585e4f50c27c11bde.osz
Using CUDA for inference (auto-selected).
Random seed: 26712
Model loaded: OliBomby/Mapperatorinator-v30 on device cuda
Using HP drain rate 5.0
Using circle size 4.0
Using overall difficulty 8.0
Using slider multiplier 1.4
Using slider tick rate 1.0
Generating map


100%|██████████| 31/31 [05:29<00:00, 10.63s/it]


Generated beatmap saved to /tmp/tmph7p21vbg/beatmapa156c94f7f43487c8a001df29167f3e9.osu
Generated .osz saved to /tmp/tmph7p21vbg/beatmap0092e312f3634d5a825434ada2d27f9a.osz


In [ ]:
run("/var/mnt/ssd/Files/Documents/Музыка/Tracks/myai - phenylacetate.mp3")

Using CUDA for inference (auto-selected).
Random seed: 11041
Model loaded: OliBomby/Mapperatorinator-v30 on device cuda
Using HP drain rate 5.0
Using circle size 4.0
Using overall difficulty 8.0
Using slider multiplier 1.4
Using slider tick rate 1.0
Generating map


100%|██████████| 48/48 [04:03<00:00,  5.07s/it]


Generated beatmap saved to /tmp/tmppkk29q_7/beatmape4475c6048d8434dbcccbe744ce0bfda.osu
Generated .osz saved to /tmp/tmppkk29q_7/beatmap89709bee67ee43deaa3ab0bb0df445f5.osz
Using CUDA for inference (auto-selected).
Random seed: 39364
Model loaded: OliBomby/Mapperatorinator-v30 on device cuda
Using HP drain rate 5.0
Using circle size 4.0
Using overall difficulty 8.0
Using slider multiplier 1.4
Using slider tick rate 1.0
Generating map


 71%|███████   | 34/48 [06:28<03:08, 13.44s/it]

In [ ]:
run("/var/mnt/ssd/Files/Documents/Музыка/Tracks/myai - anxiety causer.mp3")

In [ ]:
run("/var/mnt/ssd/Files/Documents/Музыка/Tracks/myai - balance.mp3")

In [ ]:
run("/var/mnt/ssd/Files/Documents/Музыка/Tracks/myai - brain damage.mp3")

In [ ]:
run("/var/mnt/ssd/Files/Documents/Музыка/Tracks/myai - I couldn't care less yesterday.mp3")

In [ ]:
run("/var/mnt/ssd/Files/Documents/Музыка/Tracks/myai - interactions.mp3")

In [ ]:
run("/var/mnt/ssd/Files/Documents/Музыка/Tracks/myai - Joywire.mp3")

In [ ]:
run("/var/mnt/ssd/Files/Documents/Музыка/Tracks/myai - juncstream.mp3")

In [ ]:
run("/var/mnt/ssd/Files/Documents/Музыка/Tracks/myai - leve1.mp3")

In [ ]:
run("/var/mnt/ssd/Files/Documents/Музыка/Tracks/myai - Lifetime.mp3")

In [ ]:
run("/var/mnt/ssd/Files/Documents/Музыка/Tracks/myai - lofi.mp3")

In [ ]:
run("/var/mnt/ssd/Files/Documents/Музыка/Tracks/myai - lomkn ibnujhvyctfrd rreswa3sedr yhujikop,l ft.mp3")

In [ ]:
run("/var/mnt/ssd/Files/Documents/Музыка/Tracks/myai - Metro VIP.mp3")

In [ ]:
run("/var/mnt/ssd/Files/Documents/Музыка/Tracks/myai - name.mp3")

In [ ]:
run("/var/mnt/ssd/Files/Documents/Музыка/Tracks/myai - necessity.mp3")

In [ ]:
run("/var/mnt/ssd/Files/Documents/Музыка/Tracks/myai - Noitadarged.mp3")

In [ ]:
run("/var/mnt/ssd/Files/Documents/Музыка/Tracks/myai - nothing..mp3")

In [ ]:
run("/var/mnt/ssd/Files/Documents/Музыка/Tracks/myai - Reaction.mp3")

In [ ]:
run("/var/mnt/ssd/Files/Documents/Музыка/Tracks/myai - Reduction.mp3")

In [ ]:
run("/var/mnt/ssd/Files/Documents/Музыка/Tracks/myai - Remains.mp3")

In [ ]:
run("/var/mnt/ssd/Files/Documents/Музыка/Tracks/myai - sine dnb.mp3")

In [ ]:
run("/var/mnt/ssd/Files/Documents/Музыка/Tracks/myai - Shake.mp3")

In [ ]:
run("/var/mnt/ssd/Files/Documents/Музыка/Tracks/myai - sud0.mp3")

In [ ]:
run("/var/mnt/ssd/Files/Documents/Музыка/Tracks/myai - The Problem of Self.mp3")

In [ ]:
run("/var/mnt/ssd/Files/Documents/Музыка/Tracks/myai - This is Bob.mp3")

In [ ]:
run("/var/mnt/ssd/Files/Documents/Музыка/Tracks/myai - Trouble.mp3")

In [ ]:
run("/var/mnt/ssd/Files/Documents/Музыка/Tracks/myai - Type.mp3")

In [ ]:
run("/var/mnt/ssd/Files/Documents/Музыка/Tracks/myai - unrelated.mp3")

In [ ]:
run("/var/mnt/ssd/Files/Documents/Музыка/Tracks/myai - Utility.mp3")

In [ ]:
run("/var/mnt/ssd/Files/Documents/Музыка/Tracks/myai - value.mp3")

In [ ]:
run("/var/mnt/ssd/Files/Documents/Музыка/Tracks/myai - Calculate.mp3")

In [ ]:
run("/var/mnt/ssd/Files/Documents/Музыка/Tracks/myai - Beat Is Missing.mp3")

In [ ]:
run("/var/mnt/ssd/Files/Documents/Музыка/Tracks/myai - Degradation.mp3")

In [ ]:
run("/var/mnt/ssd/Files/Documents/Музыка/Tracks/myai - Far.mp3")

In [ ]:
run("/var/mnt/ssd/Files/Documents/Музыка/Tracks/myai - fried.mp3")

In [ ]:
run("/var/mnt/ssd/Files/Documents/Музыка/Tracks/myai - light.mp3")

In [ ]:
run("/var/mnt/ssd/Files/Documents/Музыка/Tracks/myai - x..mp3")

In [ ]:
run("/var/mnt/ssd/Files/Documents/Музыка/Tracks/myai - racist.mp3")

In [ ]:
run("/var/mnt/ssd/Files/Documents/Музыка/Tracks/myai - uxelpmoc.mp3")